# 离散水印：生成轨迹可写性诊断
目的：判断更早步骤是否有实际内容选择空间，区分 MASK 随机性。

仅原两提示/种子、无水印重放；记录 t=99/75/50/25/10/5/1/0。不会训练或写入水印。
复用已跑通的模型缓存和加载兼容处理。CPU 22 项检查通过，真实 GPU 轨迹运行待验证。
选择 GPU 运行时后“全部运行”，授权挂载 Drive。结果写入新目录；预留约0.6GB诊断空间。
不同步骤的容量上界不能相加为最终容量，找到高熵步骤也不等于水印可检测。


In [ ]:
import base64, io, zipfile
from pathlib import Path
task_dir = Path('/content/ceg_discrete_pair_trajectory_v1')
task_dir.mkdir(exist_ok=True)
payload = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as archive:
    archive.extractall(task_dir)
print((task_dir/'TRAJECTORY_PROTOCOL.md').read_text())


### 安装固定依赖并检查记录器

In [ ]:
import sys
sys.path.insert(0,str(task_dir))
from process_log import run_logged
run_logged([sys.executable,'-u','-m','pip','install','--quiet',
            'diffusers==0.39.0','transformers==4.57.6','accelerate','sentencepiece',
            'safetensors','pytest','pandas','matplotlib'])
run_logged([sys.executable,'-u','-m','pytest','-q','test_coupling.py','test_capacity_interface.py',
            'test_process_log.py','test_trajectory.py'],cwd=task_dir)


### 无水印重放：两源、八个记录时刻

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import datetime, os, json
drive_root = Path('/content/drive/MyDrive/CEG-WM/discrete-pair-v1')
drive_root.mkdir(parents=True,exist_ok=True)
os.environ['HF_HOME'] = str(drive_root/'model-cache')
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
run_dir = drive_root/'runs'/('trajectory-canary-'+stamp)
reference = drive_root/'runs'/'capacity-canary-20261126T145926559663Z'
print('Output:',run_dir)
print('Log:',run_dir.with_suffix('.log'))
try:
    run_logged([sys.executable,'-u','trajectory_runner.py','--output',str(run_dir),
                '--reference-dir',str(reference)],cwd=task_dir,log_path=run_dir.with_suffix('.log'))
finally:
    if (run_dir/'status.json').exists():
        status = json.loads((run_dir/'status.json').read_text())
        print(json.dumps({k:status.get(k) for k in ['decision','stop_reason','error','wall_seconds']},indent=2))


### 结果：MASK、内容选择与局部容量
以下只显示实际完成步骤；未完成源保留在 status.json。结果需人工审阅，不自动扩展实验。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
status = json.loads((run_dir/'status.json').read_text())
rows = [dict(source_id=source['source_id'],**step) for source in status['statuses'] for step in source['steps']]
print('Sources:',[(s['source_id'],s['status']) for s in status['statuses']])
if rows:
    table = pd.DataFrame(rows)
    columns = ['source_id','timestep','incoming_mask_count','newly_unmasked',
               'mean_mask_entropy_nats','mean_probability_weighted_content_entropy_nats',
               'full_kernel_fixed16_top2_cycle_kl_upper','content_only_fixed16_top2_cycle_kl_upper']
    display(table[columns])
    table.to_csv(run_dir/'trajectory_summary.csv',index=False)
    fig,axes = plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for source,part in table.groupby('source_id',sort=False):
        part = part.sort_values('timestep',ascending=False)
        axes[0].plot(part.timestep,part.mean_mask_entropy_nats,'o--',label=source+' MASK entropy')
        axes[0].plot(part.timestep,part.mean_probability_weighted_content_entropy_nats,'s-',label=source+' weighted content')
        axes[1].plot(part.timestep,part.full_kernel_fixed16_top2_cycle_kl_upper,'o--',label=source+' full kernel')
        axes[1].plot(part.timestep,part.content_only_fixed16_top2_cycle_kl_upper,'s-',label=source+' content only')
    axes[0].set(title='Actual transition: entropy decomposition',ylabel='Mean entropy (nats / position)')
    axes[1].axhline(1.614463080360851,color='black',linestyle=':',label='Reference necessary KL')
    axes[1].set(title='Fixed 16 pairs: local upper bounds',ylabel='KL upper bound (nats)')
    axes[1].set_yscale('symlog',linthresh=1e-5)
    for ax in axes:
        ax.set_xlabel('Timestep (generation proceeds to 0)')
        ax.invert_xaxis()
        ax.grid(alpha=.2)
        ax.legend(fontsize=7)
    fig.savefig(run_dir/'trajectory_diagnostics.png',dpi=160)
    plt.show()
    print('Review data:',json.dumps(rows,ensure_ascii=False))
print('Stop here: no watermark or detection result; review the saved trajectory before any next experiment.')
